# 0.01 · Exploración inicial de datos MiBici

**Fase 0 (exploración)** según la convención de [Cookiecutter Data Science](https://cookiecutter-data-science.drivendata.org/using-the-template/): `PHASE.NOTEBOOK-iniciales-descripción`.

Objetivo:
1. Poblar `data/raw/mibici/` con los CSV de datos abiertos de MiBici (sin modificar).
2. Obtener un primer panorama: inventario, esquema por archivo, tipos, nulos y duplicados.

Este notebook **solo lee** de `data/raw`; no escribe en `interim` ni `processed`. La lógica de descarga vive en `team2_mlops2026.data.raw_data_extraction`.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
from pathlib import Path

import pandas as pd

from team2_mlops2026.data.raw_data_extraction import MANIFEST_NAME, RAW_MIBICI_DIR, ingest

pd.set_option("display.max_columns", 50)

2026-10-20 20:16:08.369 | INFO     | team2_mlops2026.config:<module>:11 - PROJ_ROOT path is: C:\MLOpsRepos\Proyecto\team2_mlops2026


## Parámetros

El histórico completo (2014 a la fecha) pesa varios GB. `SINCE` limita la descarga a partir de un mes (`YYYY-MM`); usa `None` para bajar todo. La descarga es idempotente: re-ejecutar solo baja lo que falta.

In [3]:
SINCE = "2025-01"  # None = histórico completo
WORKERS = 4  # descargas en paralelo
FORCE = False  # True = re-descargar aunque ya exista

## 1. Descarga a `data/raw`

In [4]:
summary = ingest(since=SINCE, workers=WORKERS, force=FORCE)

print(f"Publicados en la página: {summary['discovered']}")
print(f"Seleccionados (SINCE={SINCE}): {summary['selected']}")
print(f"Nuevos: {len(summary['new'])} | Ya existentes: {summary['skipped']} | Fallidos: {len(summary['failed'])}")
if summary["failed"]:
    print("Re-ejecuta la celda para reintentar:", *summary["failed"], sep="\n  ")

2026-10-20 20:16:31.649 | INFO     | team2_mlops2026.data.raw_data_extraction:ingest:175 - Encontrados 143 archivos en la página


Descargando: 100%|██████████| 22/22 [00:39<00:00,  1.79s/it]

2026-10-20 20:17:11.138 | SUCCESS  | team2_mlops2026.data.raw_data_extraction:ingest:219 - Listo: 22 nuevos, 0 omitidos, 0 fallidos
Publicados en la página: 143
Seleccionados (SINCE=2025-01): 22
Nuevos: 22 | Ya existentes: 0 | Fallidos: 0


## 2. Inventario de `data/raw/mibici`

A partir del `manifest.json` (linaje: URL de origen, sha256, tamaño, fecha de descarga).

In [5]:
manifest = json.loads((RAW_MIBICI_DIR / MANIFEST_NAME).read_text(encoding="utf-8"))

inventory = (
    pd.DataFrame.from_dict(manifest, orient="index")
    .rename_axis("name")
    .reset_index()
    .assign(
        kind=lambda d: d["name"].str.extract(r"^(datos_abiertos|nomenclatura)", expand=False),
        period=lambda d: d["name"].str.extract(r"(\d{4}_\d{2})", expand=False),
        mb=lambda d: (d["bytes"] / 2**20).round(1),
    )
    .sort_values("name")
)
inventory[["name", "kind", "period", "mb", "downloaded_at"]]

,name,kind,period,mb,downloaded_at
0,datos_abiertos_2025_01.csv,datos_abiertos,2025_01,31.4,2026-10-21T02:16:35.014491+00:00
5,datos_abiertos_2025_02.csv,datos_abiertos,2025_02,30.8,2026-10-21T02:16:46.132524+00:00
8,datos_abiertos_2025_03.csv,datos_abiertos,2025_03,33.5,2026-10-21T02:16:50.425983+00:00
1,datos_abiertos_2025_04.csv,datos_abiertos,2025_04,30.3,2026-10-21T02:16:39.290879+00:00
2,datos_abiertos_2025_05.csv,datos_abiertos,2025_05,32.7,2026-10-21T02:16:40.121420+00:00
3,datos_abiertos_2025_06.csv,datos_abiertos,2025_06,29.5,2026-10-21T02:16:42.389199+00:00
6,datos_abiertos_2025_07.csv,datos_abiertos,2025_07,30.5,2026-10-21T02:16:48.180970+00:00
4,datos_abiertos_2025_08.csv,datos_abiertos,2025_08,31.0,2026-10-21T02:16:45.503980+00:00
7,datos_abiertos_2025_09.csv,datos_abiertos,2025_09,31.2,2026-10-21T02:16:48.749947+00:00
13,datos_abiertos_2025_10.csv,datos_abiertos,2025_10,34.8,2026-10-21T02:16:59.424285+00:00


In [6]:
inventory.groupby("kind").agg(archivos=("name", "size"), mb_total=("mb", "sum"))

,archivos,mb_total
kind,,
datos_abiertos,21,653.1
nomenclatura,1,0.0


## 3. Esquema por archivo (solo encabezados)

Leer únicamente el encabezado es barato y permite detectar cambios de columnas o codificación entre meses antes de cargar datos completos.

In [7]:
def read_raw_csv(path: Path, **kwargs) -> pd.DataFrame:
    """Lee un CSV crudo probando UTF-8 y luego Latin-1 (MiBici no es consistente)."""
    try:
        return pd.read_csv(path, encoding="utf-8", **kwargs)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin-1", **kwargs)


trip_files = sorted(RAW_MIBICI_DIR.glob("datos_abiertos_*.csv"))
station_files = sorted(RAW_MIBICI_DIR.glob("nomenclatura_*.csv"))

schemas = pd.DataFrame(
    {"name": p.name, "columns": tuple(read_raw_csv(p, nrows=0).columns)} for p in trip_files
)
schemas.groupby("columns")["name"].agg(["size", "first", "last"])

,size,first,last
columns,,,
"(Viaje_Id, Usuario_Id, Genero, Año_de_nacimiento, Inicio_del_viaje, Fin_del_viaje, Origen_Id, Destino_Id)",21,datos_abiertos_2025_01.csv,datos_abiertos_2026_09.csv


## 4. Viajes: mes más reciente

In [8]:
latest_trips = trip_files[-1]
trips = read_raw_csv(latest_trips)
print(latest_trips.name, trips.shape)
trips.head()

datos_abiertos_2026_09.csv (490056, 8)


,Viaje_Id,Usuario_Id,Genero,Año_de_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id
0,45705294,4722545,M,1997.0,2026-09-01 00:00:59,2026-09-01 00:18:55,283,49
1,45705295,2344334,M,2004.0,2026-09-01 00:01:36,2026-09-01 00:17:52,67,383
2,45705301,4454420,M,2001.0,2026-09-01 00:03:36,2026-09-01 00:19:31,395,357
3,45705303,4512715,M,1996.0,2026-09-01 00:03:59,2026-09-01 00:07:07,371,57
4,45705304,2765763,M,2002.0,2026-09-01 00:04:09,2026-09-01 00:30:19,357,80


In [9]:
trips.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 490056 entries, 0 to 490055
Data columns (total 8 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Viaje_Id           490056 non-null  int64  
 1   Usuario_Id         490056 non-null  int64  
 2   Genero             486761 non-null  str    
 3   Año_de_nacimiento  490047 non-null  float64
 4   Inicio_del_viaje   490056 non-null  str    
 5   Fin_del_viaje      490056 non-null  str    
 6   Origen_Id          490056 non-null  int64  
 7   Destino_Id         490056 non-null  int64  
dtypes: float64(1), int64(4), str(3)
memory usage: 105.6 MB


In [10]:
print(f"Filas duplicadas: {trips.duplicated().sum()}")
pd.DataFrame(
    {
        "nulos": trips.isna().sum(),
        "pct_nulos": (trips.isna().mean() * 100).round(2),
        "unicos": trips.nunique(),
    }
)

Filas duplicadas: 758


,nulos,pct_nulos,unicos
Viaje_Id,0,0.00,489298
Usuario_Id,0,0.00,17395
Genero,3295,0.67,2
Año_de_nacimiento,9,0.00,70
Inicio_del_viaje,0,0.00,420403
Fin_del_viaje,0,0.00,421749
Origen_Id,0,0.00,468
Destino_Id,0,0.00,468


In [11]:
trips.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Viaje_Id,490056.0,NaN,NaN,NaN,45983257.92125,158323.729469,45705294.0,45846649.75,45984666.5,46120619.25,46255558.0
Usuario_Id,490056.0,NaN,NaN,NaN,2925815.186734,1850252.811739,102.0,754526.0,4027276.0,4655574.0,5006315.0
Genero,486761,2,M,352032,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Año_de_nacimiento,490047.0,NaN,NaN,NaN,1991.28747,13.321525,199.0,1985.0,1994.0,2000.0,2010.0
Inicio_del_viaje,490056,420403,2026-09-21 19:02:53,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Fin_del_viaje,490056,421749,2026-09-11 14:29:14,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Origen_Id,490056.0,NaN,NaN,NaN,182.159123,136.83188,2.0,53.0,170.0,276.0,519.0
Destino_Id,490056.0,NaN,NaN,NaN,182.999194,137.751268,2.0,52.0,169.0,278.0,519.0


## 5. Estaciones: nomenclatura más reciente

In [12]:
latest_stations = station_files[-1]
stations = read_raw_csv(latest_stations)
print(latest_stations.name, stations.shape)
print(f"Filas duplicadas: {stations.duplicated().sum()}")
stations.head()

nomenclatura_2026_09.csv (484, 5)
Filas duplicadas: 0


,id,name,latitude,longitude,dpcapacity
0,2,(GDL-001) C. Epigmenio Glez./ Av. 16 de Sept.,20.666378,-103.348820,15
1,3,(GDL-002) C. Colonias / Av. Niños héroes,20.667228,-103.366000,15
2,4,(GDL-003) C. Vidrio / Av. Chapultepec,20.667690,-103.368252,19
3,5,(GDL-004) C. Ghilardi /C. Miraflores,20.691847,-103.362549,19
4,6,(GDL-005) C. San Diego /Calzada Independencia,20.681158,-103.339363,11


In [13]:
stations.isna().sum().to_frame("nulos").T

,id,name,latitude,longitude,dpcapacity
nulos,0,0,0,0,0


## Hallazgos y siguientes pasos

- _Registrar aquí lo observado (cambios de esquema entre años, codificación, nulos, duplicados)._
- Siguiente notebook (fase 1): normalización de nombres hacia `data/interim` con `team2_mlops2026.data.raw_data_normalization` y limpieza.